# LifeMate — α-UMi / Qwen3-4B GLPFT
Shared 4B backbone; global QLoRA → three independent local continuations. Qwen generates train-only rationales with prefix-only observations. Validation compares global vs local using oracle text and audio. Public test is inference-only. Pilot is a limited training experiment; full follows the paper's epoch schedule.


In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib, zipfile

REPO_URL = "https://github.com/tnisl/baseline.git"
GIT_REF = "solution_paper"
PROFILE = "pilot"  # smoke: 3 updates/stage; pilot: 80+40+40+40; full: 2/1/1/2 epochs
SOURCE = Path("/kaggle/working/paper_source")
WORK = Path("/kaggle/working/paper_run") / PROFILE
if not SOURCE.exists():
    subprocess.run(["git","clone","--filter=blob:none","--no-checkout",REPO_URL,str(SOURCE)],check=True)
subprocess.run(["git","-C",str(SOURCE),"fetch","origin",GIT_REF],check=True)
subprocess.run(["git","-C",str(SOURCE),"checkout","--detach","FETCH_HEAD"],check=True)
SOURCE_COMMIT=subprocess.check_output(["git","-C",str(SOURCE),"rev-parse","HEAD"],text=True).strip()
print("Source commit:",SOURCE_COMMIT)


In [ ]:
# Keep Kaggle's CUDA-matched PyTorch.
subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check",
                "-r",str(SOURCE/"requirements.txt")],check=True)


In [ ]:
# Discover an already attached copy; competition data is never uploaded to GitHub.
roots=[p.parent.parent for p in Path("/kaggle/input").rglob("train.jsonl")
       if p.parent.name=="train" and (p.parent.parent/"public_test/public_test.jsonl").exists()]
if not roots:
    archives=list(Path("/kaggle/input").rglob("train+public_test.zip"))
    if not archives: raise FileNotFoundError("Attach the authorised LifeMate train+public_test dataset.")
    destination=Path("/kaggle/working/competition_data");destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target=(destination/member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):raise ValueError("Unsafe archive member")
        archive.extractall(destination)
    roots=[p.parent.parent for p in destination.rglob("train.jsonl") if p.parent.name=="train"]
if len(roots)!=1:raise ValueError(f"Expected one data root, found: {roots}")
DATA_ROOT=roots[0]
expected=json.loads((SOURCE/"docs/data_audit.json").read_text())["train_sha256"]
actual=hashlib.sha256((DATA_ROOT/"train/train.jsonl").read_bytes()).hexdigest()
assert actual==expected, "Attached dataset differs from audited release"
WORK.mkdir(parents=True,exist_ok=True)
(WORK/"source.json").write_text(json.dumps({"source_commit":SOURCE_COMMIT,"train_sha256":actual}))
print("Dataset:",DATA_ROOT)


In [ ]:
# Fresh processes release GPU memory between rationale/global/local/validation phases.
subprocess.run([sys.executable,"-u",str(SOURCE/"finetune.py"),
    "--data-root",str(DATA_ROOT),"--output-dir",str(WORK),"--profile",PROFILE,
    "--model","Qwen/Qwen3-4B-Instruct-2507","--eval-limit","20"],
    cwd=SOURCE,check=True)


In [ ]:
status=json.loads((WORK/"status.json").read_text())
assert status["status"]=="complete",status
assert status["stages"]==["global","planner","caller","summarizer"]
global_meta=json.loads((WORK/"adapters/global/lineage.json").read_text())
for role in ("planner","caller","summarizer"):
    meta=json.loads((WORK/f"adapters/{role}/lineage.json").read_text())
    assert meta["initial_adapter"]["sha256"]==global_meta["final_adapter"]["sha256"]
    assert meta["final_adapter"]["finite"]
    assert meta["initial_adapter"]["sha256"]!=meta["final_adapter"]["sha256"]
for path in sorted(WORK.glob("*.metrics.json")):
    result=json.loads(path.read_text())
    print(path.name,{k:v for k,v in result.items() if k!="per_sample"})
print("Private notebook artifacts:",WORK)
